# Práctica — Ejercicio 1: Flete aéreo de vacunas COVID-19

El dataset `flete-aereo-vacunas-covid19-al-2021-06-28.xlsx` contiene información sobre los fletes aéreos
realizados para transportar vacunas contra el COVID-19 hasta Argentina.

---

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# El archivo tiene 4 filas de metadatos antes del encabezado real
df = pd.read_excel("../Datasets/flete-aereo-vacunas-covid19-al-2021-06-28.xlsx", header=4)

# Eliminar filas vacías (el Excel usa todas las filas de la hoja)
df = df.dropna(subset=[df.columns[0]])

# Limpiar espacios en nombres de columna y renombrar para evitar caracteres especiales
df.columns = [c.strip() for c in df.columns]
df.columns = [
    'organismo', 'expediente', 'financiamiento', 'acto_conclusion',
    'descripcion', 'prestador', 'CUIT', 'estado', 'factura_nro',
    'factura_moneda_monto', 'guia_nro', 'guia_moneda_monto',
    'fecha_guia', 'vuelo', 'bienes_transportados',
    'proveedor_bienes_transportados', 'operacion', 'comprador_donante', 'origen'
]

# Convertir fecha_guia a datetime
df['fecha_guia'] = pd.to_datetime(df['fecha_guia'])

print(f"Registros: {len(df)}")
df.head(3)

## Ítem 1 — Vuelo con mayor cantidad de fletes

> **¿Cuál fue el número de vuelo que realizó una mayor cantidad de fletes?**

In [ ]:
conteo_vuelos = df['vuelo'].value_counts()
print("Top 5 vuelos por cantidad de fletes:")
print(conteo_vuelos.head())
print(f"\nVuelo con más fletes: {conteo_vuelos.idxmax()} ({conteo_vuelos.max()} fletes)")

## Ítem 2 — Registros sin número de vuelo

> **¿Cuántos registros no contienen información sobre el número de vuelo?**

In [ ]:
sin_vuelo = df['vuelo'].isna().sum()
print(f"Registros sin número de vuelo: {sin_vuelo}")

## Ítem 3 — Promedio facturado en USD

> **Calcule el promedio de lo facturado en todos los vuelos realizados considerando únicamente los registros de viajes que se facturaron en USD.**

In [ ]:
# Filtrar solo registros en USD
mask_usd = df['factura_moneda_monto'].str.startswith('USD', na=False)
df_usd = df[mask_usd].copy()

# Parsear el monto: 'USD 305.250,00' → 305250.00
def parsear_monto(s):
    num = str(s).replace('USD', '').strip()
    num = num.replace(' ', '').replace('.', '').replace(',', '.')
    try:
        return float(num)
    except ValueError:
        return float('nan')

df_usd['monto'] = df_usd['factura_moneda_monto'].apply(parsear_monto)
promedio_usd = df_usd['monto'].mean()

print(f"Registros facturados en USD: {len(df_usd)}")
print(f"Promedio facturado (USD): {promedio_usd:,.2f}")

## Ítem 4 — Porcentaje de vuelos desde Rusia o China

> **¿Qué porcentaje de los vuelos realizados tuvieron como origen a Rusia o a China?**

In [ ]:
rusia_china = df['origen'].str.lower().str.strip().isin(['rusia', 'china'])
pct = rusia_china.sum() / len(df) * 100

print("Distribución de orígenes:")
print(df['origen'].value_counts())
print(f"\nVuelos desde Rusia o China: {rusia_china.sum()} de {len(df)}")
print(f"Porcentaje: {pct:.1f}%")

## Ítem 5 — Vuelo más reciente y días entre primer y último vuelo

> **¿Cuál es el vuelo más reciente de los que se tiene registro en el dataset? ¿Cuántos días transcurrieron entre el primer y el último vuelo realizados?**

In [ ]:
primer_vuelo = df['fecha_guia'].min()
ultimo_vuelo = df['fecha_guia'].max()
dias = (ultimo_vuelo - primer_vuelo).days

print(f"Primer vuelo: {primer_vuelo.date()}")
print(f"Último vuelo: {ultimo_vuelo.date()}")
print(f"Días transcurridos entre el primero y el último: {dias} días")

# Registro del vuelo más reciente
print("\nRegistro más reciente:")
df[df['fecha_guia'] == ultimo_vuelo][['fecha_guia', 'vuelo', 'origen', 'bienes_transportados']].head()

## Ítem 6 — Exportar como parquet

> **Escriba el archivo en formato .parquet.**

In [ ]:
# Convertir vuelo a string para compatibilidad con parquet (tiene valores mixtos)
df['vuelo'] = df['vuelo'].astype(str)
ruta_parquet = "../Datasets/flete_vacunas.parquet"
df.to_parquet(ruta_parquet, index=False)
print(f"Archivo guardado como parquet: {ruta_parquet}")

# Verificar que se puede leer correctamente
df_check = pd.read_parquet(ruta_parquet)
print(f"Verificación — Shape: {df_check.shape}")
df_check.head(2)

## Conclusiones

- **Ítem 1:** El vuelo **1061** realizó la mayor cantidad de fletes (15 vuelos).
- **Ítem 2:** Hay **3 registros** sin número de vuelo.
- **Ítem 3:** El promedio facturado en USD fue de aproximadamente **USD 206.827** (varía según los registros con monto válido).
- **Ítem 4:** El **98.5%** de los vuelos tuvieron origen en Rusia o China (65 de 66 registros).
- **Ítem 5:** El vuelo más reciente fue el **22 de junio de 2021**. Entre el primer y último vuelo transcurrieron **181 días**.
- **Ítem 6:** El dataset fue exportado correctamente al formato `.parquet`.